# Self-Supervised Spectral Foundation Model (SFM)
## Validation & Model Demonstration Notebook

This notebook provides an interactive walkthrough of the core components of the Spectral Foundation Model pipeline for multi-gas Fourier Transform Infrared (FTIR) sensing:
1. **Physical FTIR Spectrum Simulation**: Beer-Lambert mixture modeling with instrumental line shape (ILS) convolution and baseline drift.
2. **1D Masked Spectral Autoencoder (MAE)**: 75% patch masking and ro-vibrational peak reconstruction.
3. **Multi-Gas Identification & Quantification**: Multi-label classification and concentration regression.
4. **Baseline Benchmark (PLSR vs. SFM)**: In-distribution and out-of-distribution (OOD) degradation comparison.
5. **Safety Interpretation Layer**: Translating predicted gas levels into actionable OSHA/NIOSH safety advisories.

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure local package is in Python path
sys.path.insert(0, str(Path.cwd().resolve() / 'src'))

from sfm.models.mae import SpectralMAE
from sfm.models.heads import MultiGasHead
from sfm.interpretation.llm import LLMInterpreter
from sfm.data.transforms import Compose, AddGaussianNoise, RandomBaselineDrift

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')

### 1. Synthetic FTIR Spectrum & Optical Perturbations
FTIR spectra in the 2000-4000 cm^-1 region exhibit sharp ro-vibrational bands. In real-world conditions, optical baseline drift and detector noise degrade the signal.

In [ ]:
# Generate a synthetic multi-gas spectrum with 257 wavenumber channels
wavenumbers = np.linspace(2000, 4000, 257)

# Simulate characteristic absorption bands (e.g. CO2 at 2350 cm^-1, CH4 at 3018 cm^-1, CO at 2143 cm^-1)
clean_spectrum = np.zeros_like(wavenumbers)
clean_spectrum += 0.85 * np.exp(-((wavenumbers - 2349) / 25)**2)  # CO2 asymmetric stretch
clean_spectrum += 0.45 * np.exp(-((wavenumbers - 3018) / 35)**2)  # CH4 v3 band
clean_spectrum += 0.30 * np.exp(-((wavenumbers - 2143) / 20)**2)  # CO fundamental

# Apply physical perturbations: baseline drift and Gaussian detector noise
augmentor = Compose([
    RandomBaselineDrift(drift_order=2, max_amplitude=0.15, p=1.0),
    AddGaussianNoise(snr_db=25.0, p=1.0)
])
degraded_spectrum = augmentor(clean_spectrum.copy())

plt.figure(figsize=(10, 4), dpi=100)
plt.plot(wavenumbers, clean_spectrum, label='Nominal Absorption Spectrum', color='#10b981', lw=2)
plt.plot(wavenumbers, degraded_spectrum, label='Degraded (Drift + Noise, SNR=25dB)', color='#ef4444', lw=1.2, alpha=0.85)
plt.xlabel('Wavenumber (cm^-1)')
plt.ylabel('Absorbance')
plt.title('Simulated Multi-Gas FTIR Spectrum: Nominal vs. Degraded')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 2. 1D Masked Spectral Autoencoder (MAE) Architecture
The foundation model employs an asymmetric encoder-decoder with 75% patch masking. Patches of size 10 cm^-1 are partitioned and randomly masked.

In [ ]:
# Initialize SpectralMAE backbone
mae = SpectralMAE(
    seq_len=257,
    patch_size=10,
    in_channels=1,
    encoder_dim=256,
    encoder_depth=6,
    encoder_heads=8,
    decoder_dim=128,
    decoder_depth=4,
    decoder_heads=4,
    mask_ratio=0.75
).to(device)

print(f'Total Model Parameters: {sum(p.numel() for p in mae.parameters()):,}')

# Forward pass with 75% random masking
x = torch.from_numpy(degraded_spectrum).float().unsqueeze(0).unsqueeze(1).to(device)
mae.eval()
with torch.no_grad():
    loss, pred, mask = mae(x)

print(f'Self-Supervised Reconstruction Loss (MSE): {loss.item():.4f}')
print(f'Mask Tensor Shape: {mask.shape} (Number of patches masked: {int(mask.sum().item())}/{mask.numel()})')

### 3. Multi-Gas Identification & Quantification Head
The frozen pre-trained encoder feeds into a multi-task head providing simultaneous multi-label gas detection (BCE) and concentration estimation (MSE).

In [ ]:
target_gases = ['CO2', 'CO', 'CH4', 'N2O', 'H2O']
head = MultiGasHead(
    encoder_dim=256,
    num_classes=len(target_gases),
    hidden_dim=128,
    dropout=0.1
).to(device)

# Simulate downstream prediction
head.eval()
with torch.no_grad():
    latent = mae.forward_features(x)  # Extract latent representations
    logits, concentrations = head(latent)
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    conc_ppm = (concentrations.squeeze(0).cpu().numpy() * 5000.0).clip(min=0.0)

print('--- Multi-Gas Downstream Predictions ---')
for gas, p, c in zip(target_gases, probs, conc_ppm):
    status = 'DETECTED' if p > 0.5 else 'NOT DETECTED'
    print(f'  [{gas:4s}] Prob: {p*100:5.1f}% | Estimated Conc: {c:7.1f} ppm | Status: {status}')

### 4. Domain LLM Safety Interpretation Layer
An edge-compatible LLM interpretation module translates numerical gas readings into structured safety advisories, checking regulatory safety thresholds (OSHA PEL, NIOSH REL).

In [ ]:
# Initialize interpreter (rule-based fallback ensures 100% deterministic edge safety)
interpreter = LLMInterpreter(model_name='local-rule', enable_local_llm=False)

sample_readings = {
    'CO2': 5500.0,   # Exceeds OSHA 8-hr TWA (5000 ppm)
    'CO': 45.0,      # Exceeds NIOSH REL (35 ppm)
    'CH4': 12000.0,  # Elevated flammable gas
    'N2O': 10.0,
    'H2O': 8500.0
}

interpretation = interpreter.interpret(sample_readings, context='Chemical warehouse ventilation audit')

print(f'Overall Severity: {interpretation.severity}')
print(f'Summary: {interpretation.summary}
')
print('Immediate Actions Required:')
for action in interpretation.actions:
    print(f'  • {action}')